## Conectarse a RDS PostgreSQL

In [1]:
import boto3
import json
from sqlalchemy import create_engine

# Recuperar credenciales desde Secrets Manager
client = boto3.client("secretsmanager", region_name="us-east-1")
secret = client.get_secret_value(SecretId="itam/rds/flights/credentials")
creds = json.loads(secret["SecretString"])

# Endpoint primario — para crear tablas e insertar datos
RDS_ENDPOINT = "itam-northwind-561137843164.cw5yuskyg5vm.us-east-1.rds.amazonaws.com"

engine = create_engine(
    f"postgresql+psycopg2://{creds['username']}:{creds['password']}@{RDS_ENDPOINT}:{creds['port']}/{creds['dbname']}"
)

with engine.connect() as conn:
    print("✓ Conexión exitosa a RDS PostgreSQL")

✓ Conexión exitosa a RDS PostgreSQL


## Definir el schema y crear las tres tablas

In [2]:
from sqlalchemy.orm import DeclarativeBase, mapped_column, Mapped
from sqlalchemy import String, Integer, Float, ForeignKey
from typing import Optional

class Base(DeclarativeBase):
    pass

class Airline(Base):
    __tablename__ = "airlines"
    iata_code:  Mapped[str]             = mapped_column(String(10),  primary_key=True)
    airline:    Mapped[str]             = mapped_column(String(100))

class Airport(Base):
    __tablename__ = "airports"
    iata_code:  Mapped[str]             = mapped_column(String(10),  primary_key=True)
    airport:    Mapped[str]             = mapped_column(String(200))
    city:       Mapped[Optional[str]]   = mapped_column(String(100))
    state:      Mapped[Optional[str]]   = mapped_column(String(50))
    country:    Mapped[Optional[str]]   = mapped_column(String(50))
    latitude:   Mapped[Optional[float]] = mapped_column(Float)
    longitude:  Mapped[Optional[float]] = mapped_column(Float)

class Flight(Base):
    __tablename__ = "flights"
    id:                  Mapped[int]             = mapped_column(Integer, primary_key=True, autoincrement=True)
    year:                Mapped[int]             = mapped_column(Integer)
    month:               Mapped[int]             = mapped_column(Integer)
    day:                 Mapped[int]             = mapped_column(Integer)
    airline:             Mapped[str]             = mapped_column(String(10), ForeignKey("airlines.iata_code"))
    origin_airport:      Mapped[str]             = mapped_column(String(10), ForeignKey("airports.iata_code"))
    destination_airport: Mapped[str]             = mapped_column(String(10), ForeignKey("airports.iata_code"))
    departure_delay:     Mapped[Optional[float]] = mapped_column(Float)
    arrival_delay:       Mapped[Optional[float]] = mapped_column(Float)
    cancelled:           Mapped[int]             = mapped_column(Integer)
    cancellation_reason: Mapped[Optional[str]]   = mapped_column(String(5))
    distance:            Mapped[Optional[float]] = mapped_column(Float)
    air_system_delay:    Mapped[Optional[float]] = mapped_column(Float)
    airline_delay:       Mapped[Optional[float]] = mapped_column(Float)
    weather_delay:       Mapped[Optional[float]] = mapped_column(Float)
    late_aircraft_delay: Mapped[Optional[float]] = mapped_column(Float)
    security_delay:      Mapped[Optional[float]] = mapped_column(Float)

# Idempotencia
Base.metadata.drop_all(engine)
Base.metadata.create_all(engine)
print("✓ Schema creado en PostgreSQL")

✓ Schema creado en PostgreSQL


## Insertar datos

In [5]:
import pandas as pd
from sqlalchemy.orm import Session
from sqlalchemy import insert


def load_csv(session, model, df):
    df = df.rename(columns=str.lower)
    records = [
        {k: None if pd.isnull(v) else v for k, v in row.items()}
        for row in df.to_dict(orient="records")
    ]
    session.execute(insert(model), records)
    print(f"✓ {model.__tablename__}: {len(records):,} filas cargadas")

df_airlines = pd.read_csv("data/flights/airlines.csv")
df_airports = pd.read_csv("data/flights/airports.csv")
df_flights  = pd.read_csv("data/flights/flights.csv", nrows=500_000)

with Session(engine) as session:
    load_csv(session, Airline, df_airlines)
    load_csv(session, Airport, df_airports)
    load_csv(session, Flight,  df_flights)
    session.commit()
    print("✓ Carga completa")

✓ airlines: 14 filas cargadas
✓ airports: 322 filas cargadas
✓ flights: 500,000 filas cargadas
✓ Carga completa
